# LeNet 与 MNIST

两次卷积提取局部特征，两次池化缩小图片，展平后交给全连接层输出十类得分。

保留原训练量：2000 张训练图、5 轮、每批 32 张；独立测试 500 张。
纯 Python CPU 训练需要时间。请在 `examples` 目录运行。

In [1]:
import sys

# 可从项目根、examples 或当前分组目录启动。
sys.path[:0] = ["..", "examples"]

from qutai import Stream, nn, visualization
from datasets import load_mnist

# 使用 CPU 时改为 "cpu"；模型与浮点输入保持相同设备和精度。
device = "cuda"
dtype = "float32"

## 1. 读取数据，看看一张图片

图像张量使用 `(批次, 通道, 高, 宽)`，分批器将像素除以 255。训练集和测试集来自官方不同划分。

In [2]:
train_count = 2000
test_count = 500
epochs = 5
batch_size = 32
learning_rate = 0.1
labels = list(range(10))

train_images, train_targets = load_mnist("train")
test_images, test_targets = load_mnist("test")
inputs, targets = next(iter(Stream(train_images, train_targets, batch_size=batch_size, limit=train_count, shuffle=False, progress=False)))
print("输入形状：", inputs.shape, "第一张的标签：", labels[targets[0]])
visualization.Image.from_tensor(inputs[0], value_range=(0, 1))

输入形状： (32, 1, 28, 28) 第一张的标签： 5


## 2. 搭建模型

两次卷积提取局部特征，两次池化缩小图片，展平后交给全连接层输出十类得分。
交叉熵接收原始得分和整数标签，训练时无需先算 Softmax。

In [3]:
model = nn.Model([
    nn.Conv2d(1, 6, 5, padding=2),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(6, 16, 5),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(400, 120),
    nn.ReLU(),
    nn.Linear(120, 84),
    nn.ReLU(),
    nn.Linear(84, 10),
]).to(device, dtype=dtype)
optimizer = nn.SGD(model, learning_rate=learning_rate)
loss_fn = nn.CrossEntropy()
visualization.Network(model)

### 观察每层如何改变形状

只取一张图，在预测状态下逐层查看；模型中的参数和 BatchNorm 统计保持原样。

In [4]:
output = inputs[:1].to(device, dtype=dtype)
for layer in model.layers:
    output = nn.Model([layer])(output, train=False)
    print(type(layer).__name__, output.shape)

Conv2d (1, 6, 28, 28)
ReLU (1, 6, 28, 28)
MaxPool2d (1, 6, 14, 14)
Conv2d (1, 16, 10, 10)
ReLU (1, 16, 10, 10)
MaxPool2d (1, 16, 5, 5)
Flatten (1, 400)
Linear (1, 120)
ReLU (1, 120)
Linear (1, 84)
ReLU (1, 84)
Linear (1, 10)


## 3. 按批训练

每轮重新打乱训练子集，然后逐批更新参数。训练循环保持与回归、Iris 相同的四个步骤；每轮打印最后一批的损失。

In [5]:
data = Stream(train_images, train_targets, batch_size=batch_size, limit=train_count)

for epoch in range(1, epochs + 1):
    for inputs, targets in data:
        inputs = inputs.to(device, dtype=dtype)
        optimizer.zero_grad()
        loss = loss_fn(model(inputs), targets)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch}/{epochs}  最后一批 loss {loss.item():.6f}")

▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰ 100.0%  63/63  2.7s
epoch 1/5  最后一批 loss 2.210651
▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰ 100.0%  63/63  2.9s
epoch 2/5  最后一批 loss 0.899053
▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰ 100.0%  63/63  2.9s
epoch 3/5  最后一批 loss 0.558075
▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰ 100.0%  63/63  3.1s
epoch 4/5  最后一批 loss 0.080279
▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰▰ 100.0%  63/63  2.9s
epoch 5/5  最后一批 loss 0.599601


## 4. 独立测试

`train=False` 暂停计算图记录；ResNet 的 BatchNorm 也会使用训练得到的统计量。直接取最大得分的编号作为类别。

In [6]:
correct, total = 0, 0
for inputs, targets in Stream(test_images, test_targets, batch_size=batch_size, limit=test_count, progress=False):
    inputs = inputs.to(device, dtype=dtype)
    predictions = model(inputs, train=False).argmax(axis=-1)
    correct += sum(prediction == target for prediction, target in zip(predictions, targets))
    total += len(targets)
print(f"测试准确率：{correct}/{total} = {correct / total:.1%}")

测试准确率：434/500 = 86.8%


In [7]:
inputs, targets = next(iter(Stream(test_images, test_targets, batch_size=4, progress=False)))
inputs = inputs.to(device, dtype=dtype)
predictions = model(inputs, train=False).argmax(axis=-1)
visualization.Table([[labels[target], labels[prediction]] for target, prediction in zip(targets, predictions)], columns=["正确类别", "预测类别"])